### Regression

1. Data cleaning ☑
2. Data split ☑
3. EDA ☑
4. Baseline model ☑
5. Features ☑
6. Linear Regression (Ridge, Lasso) ☑ 
7. Polynomial Regression ☑

In [ ]:
# Library

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import sklearn.pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
# Load data

river_avg = pd.read_csv("data/avg_river_levels.csv")
river_min = pd.read_csv("data/min_river_levels.csv")
river_max = pd.read_csv("data/max_river_levels.csv")

river_location = pd.read_csv("data/monitoring_locations.csv")

In [ ]:
# Function 1: Reshape + merge df

def reshape_long(avg_df, min_df, max_df, loc_df, start = "2017-04-01", drop_tidal = True):
    
    """
    INPUT:
    - avg_df, min_df, max_df: Average, Minimum and Maximum df
    - loc_df: locations df
    - start: observations before this date will be removed
    - drop_tidal: boolean value, should tidal-level stations be removed
    
    OUTPUT: 
    - river long df: df with one row per station per day and columns
    """
    
    data = [avg_df, min_df, max_df]
    labels = ["avg", "min", "max"]
    column = ["County", "Station_Name", "Observation_ID", "date"]
    start  = pd.Timestamp(start)
    long_df = {}

    for df, lab in zip(data, labels):
        if drop_tidal:
            df = df[~df["Observation_ID"].str.contains("tidal-level", na = False)]  
        
        long_df[lab] = df.melt(id_vars = column[0:3], var_name = "date", value_name = lab)
        long_df[lab]["date"] = pd.to_datetime(long_df[lab]["date"], format = "%d/%m/%Y")
        long_df[lab] = long_df[lab][long_df[lab]["date"] >= start]

    river_long = (long_df["avg"]
                 .merge(long_df["min"], on = column, how = "left")
                 .merge(long_df["max"], on = column, how = "left")
                 .merge(loc_df, on = column[0:3], how = "left"))

    for col in column[0:3]:
        river_long[col] = river_long[col].astype("category")
    
    return river_long

In [ ]:
# Function 2: Clean error in long_df

def clean_error(df, k = 20, cutoff = "2026-08-31"):
    
    """
    INPUT: 
    - df: long df that contains Observation_ID, date, avg, min, max, latitude, longitude
    - k: number of IQRs beyond Q1/Q3 at which a value is an outlier
    - cutoff: last day of training period
    
    OUTPUT:
    - df2 : a copy of the df with the bad values replaced by NaN values
    """
    
    labels = ["avg", "min", "max"]
    cutoff = pd.Timestamp(cutoff) 
    df2 = df.copy()

    err = (df2[labels].isin([99999.999, -999999.999, -99]) | (df2[labels].abs() >= 9999)).any(axis = 1)
    df2.attrs["error"] = int(err.sum())
    df2.loc[err, labels] = np.nan
             
    for lab in labels:
        train_val = df2[lab].where(df2["date"] <= cutoff)
        per_sta = train_val.groupby(df2["Observation_ID"], observed = True)
        
        q1 = per_sta.transform(lambda s: s.quantile(0.25))
        q3 = per_sta.transform(lambda s: s.quantile(0.75))

        sd = per_sta.transform("std")
        iqr = (q3 - q1).where(q3 - q1 > 0, sd)  # if iqr=0, use sd
        
        outlier = ((df2[lab] > q3 + k * iqr) | (df2[lab] < q1 - k * iqr))
        df2.attrs[f"outliers_{lab}"] = int(outlier.sum())        
        df2.loc[outlier, lab] = np.nan

    # blank 'impossible' days
    bad = (df2["min"] > df2["avg"]) | (df2["avg"] > df2["max"])
    df2.attrs["order_violations"] = int(bad.sum())
    df2.loc[bad, labels] = np.nan
    
    return df2

In [ ]:
# Function 3: drop IDs in clean_df

def drop_ids(df, min_years = 3, cutoff = "2026-08-31"):
    
    """
    INPUT:
    - df (long+cleaned dataframe): contains Observation_ID, date, avg, min, max, latitude, longitude
    - min_years: minimum number of years of non missing avg data a station needs to be kept
    - cutoff: data that marks the end of the period (given from the question)
    
    OUTPUT:
    - keep_df: a copy of df containing only stations we keep
    """

    cutoff = pd.Timestamp(cutoff) 
    
    train_obs = df["avg"].notna() & (df["date"] <= pd.Timestamp(cutoff))
    years = train_obs.groupby(df["Observation_ID"], observed = True).sum() / 365.25

    keep_ids = years[years >= min_years].index

    keep_df = df[df["Observation_ID"].isin(keep_ids)].copy()
    keep_df.attrs["dropped_ids"] = int((years < min_years).sum())
    keep_df.attrs["kept_ids"] = int((years >= min_years).sum())

    return keep_df

In [ ]:
# Function 4: Split df to train/test 

def make_split(df, cutoff, end, feature_cols, target):

    """
    INPUT:
    - clean_df: long df containing  Observation_ID, date, feature columns and target column
    - validation: cutoff = "2025-08-31", end = "2025-11-30"
    - final test: cutoff = "2026-08-31", end = "2026-11-30"
    - feature_cols: list of features names
    - target: target name to predict
    
    OUTPUT:
    - X_train, y_train, X_test, y_test
    """
    
    cutoff, end = pd.Timestamp(cutoff), pd.Timestamp(end)
    cols = feature_cols + [target] 

    train_df = (df[df["date"] <= cutoff]
               .dropna(subset = cols)
               .sort_values(["Observation_ID", "date"]))
    val_df = (df[(df["date"] > cutoff) & (df["date"] <= end)]
             .dropna(subset = cols)
             .sort_values(["Observation_ID", "date"]))
    

    X_train, y_train = train_df[feature_cols], train_df[target]
    X_test,  y_test  = val_df[feature_cols],  val_df[target]

    return X_train, y_train, X_test, y_test

In [ ]:
river_long = reshape_long(river_avg, river_min, river_max, river_location)
river_clean = clean_error(river_long)
river_df = drop_ids(river_clean)

river_df.attrs

In [ ]:
### Data overview
labels = ["avg", "min", "max"]

cutoff_val = "2025-08-31"; end_val = "2025-11-30"      # validation
cutoff_final = "2026-08-31"; end_final = "2026-11-30"  # final test

# train_df 
train_df = river_df[river_df["date"] <= cutoff_final].copy()

train_df.info()                # -> inspect column names, types, non-null counts, memory
train_df[labels].describe()    # -> summary stats for avg/min/max
train_df[labels].isna().sum()  # -> NaNs in avg/min/max

print("Stations:", train_df["Observation_ID"].nunique())        # number of obs_id 
print("Counties:", train_df["County"].nunique())                # number of counties 
print("Dates:", train_df["date"].min().date(),  # first day 
      "to", train_df["date"].max().date())      # last date

In [ ]:
# avg level stats for each month (seasonal pattern)

train_df["month"] = train_df["date"].dt.month
train_df.groupby("month")["avg"].describe()

In [ ]:
# train_df (each station's days in time order)
train_df = train_df.sort_values(["Observation_ID", "date"]).copy()
per_sta = train_df.groupby("Observation_ID", observed=True)["avg"]  # avg grouped by station

# z-score per station: (today's level − that station's mean) / that station's std
train_df["z"] = (train_df["avg"] - per_sta.transform("mean")) / per_sta.transform("std")

train_df["year"] = train_df["date"].dt.year  # 2017, 2018, ...
train_df["month"] = train_df["date"].dt.month  # 1–12
train_df["doy"] = train_df["date"].dt.dayofyear  # day of year, 1–366

In [ ]:
# sample avg_df (with no NaNs)
sample = train_df.dropna(subset = ["avg"]).sample(5000, random_state = 0)

sample["season"] = sample["month"].map({6: "Summer", 7: "Summer", 8: "Summer",    # label Jun–Aug as Summer
                                        9: "Autumn", 10: "Autumn", 11: "Autumn"}) # label Sep–Nov as Autumn (forecast period)
                                                                                  # ... other months -> NaN

# daily range
sample["daily_range"] = sample["max"] - sample["min"]

# autumn
autumn = sample[sample["season"] == "Autumn"]

In [ ]:
# Correlation table
train_df[["avg", "min", "max", "Latitude", "Longitude"]].corr()["avg"]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize = (18, 5))     


# (left) z per month (boxplot)
sns.boxplot(data = train_df, x = "month", y = "z",
            showfliers = False,  # outlier dots hidden
            ax = axes[0])
axes[0].set_title("Standardized level (per month)")

# (middle) z per season: is forecast period different from the end of training?
sns.violinplot(data = sample.dropna(subset = ["season"]), x = "season", y = "z",
               cut = 0, order = ["Summer", "Autumn"], ax = axes[1])
axes[1].set_ylim(-4, 8)
axes[1].set_title("Summer (training end) vs. Autumn (forecast period)")  

# (right) autumn per year: how much does autumn vary from year to year?
sns.boxplot(data = autumn, x = "year", y = "z", showfliers = False, ax = axes[2])
axes[2].set_title("September to November (per year)")


fig.suptitle("EDA (training data)", fontsize = 14)
plt.tight_layout()
plt.show()

In [ ]:
# get_coefs() function from lab 5
def get_coefs(m, plot = False, feature_names = None, figsize = (5,5), figtitle = None, intercept = True):
    
    """   
    INPUT:
    - m: sklearn LinearRegression model object (or pipeline with LinearRegression as final step)
    - plot: boolean value, should coefficients be plotted with error bars
    - feature_names: list of feature names to use in the plot 
    - figsize: tuple defining figure size
    - figtitle: string defining figure title
    - intercept: boolean value, should intercept be included in the plot

    OUTPUT:
    - Model coefficients in df for fitted linear model
    """
    
    # Extract intercept and coefficients into single array
    w = np.concatenate(([m[-1].intercept_] if isinstance(m, sklearn.pipeline.Pipeline) else [m.intercept_], 
                        m[-1].coef_ if isinstance(m, sklearn.pipeline.Pipeline) else m.coef_))
    
    # Extract name of features
    if feature_names is None:
        feature_names = m[:-1].get_feature_names_out() if isinstance(m, sklearn.pipeline.Pipeline) else m.feature_names_in_
        feature_names = np.concatenate((['intercept'], feature_names))
        
    # Create df
    w_df = pd.DataFrame({'feature': feature_names, 'coef': w}).sort_values ("coef", ascending = False)

    if plot:
        if not intercept:
            w_df = w_df[w_df['feature'] != 'intercept']
            
        plt.figure(figsize = figsize); plt.barh(w_df['feature'], w_df['coef'])
        plt.ylabel('Features'); plt.xlabel('Coefficient Value')
        plt.axvline(x = 0, color = ".5")
        
        if figtitle is not None:
            plt.title(figtitle)
            
        plt.grid(); plt.show()
    
    return  w_df

In [1]:
# cos/sin for periodicity
def add_features(df, cutoff):
    
    """
    INPUT:
    - df: long df containing Observation_ID, date, avg, min, max
    - cutoff: last training date ("2025-08-31" for validation, "2026-08-31" for final test)

    OUTPUT:
    - df2: copy of df with columns added (station stats use training rows only):
        * sta_mean, sta_std: station's mean and stdev of avg level
        * z: standardized level (target)
        * daily_range: station's median (max - min), divided by its IQR of avg
        * sin_doy, cos_doy (+ sin2_doy, cos2_doy): day-of-year waves
    """
    
    df2 = df.copy()  
    cutoff = pd.Timestamp(cutoff)
    is_train = df2["date"] <= cutoff 

    train_val = df2["avg"].where(is_train)                 
    per_sta = train_val.groupby(df2["Observation_ID"], observed=True)

    df2["sta_mean"] = per_sta.transform("mean")
    df2["sta_std"]  = per_sta.transform("std").replace(0, np.nan)   # stdev 0 → NaN (avoid ±∞)
    df2["z"] = (df2["avg"] - df2["sta_mean"]) / df2["sta_std"]

    rng_train = (df2["max"] - df2["min"]).where(is_train)
    q1 = per_sta.transform(lambda s: s.quantile(0.25))
    q3 = per_sta.transform(lambda s: s.quantile(0.75))
    iqr = (q3 - q1).replace(0, np.nan)  # IQR 0 → NaN
    df2["daily_range"] = rng_train.groupby(df2["Observation_ID"], observed=True).transform("median") / iqr

    theta = 2 * np.pi * df2["date"].dt.dayofyear / 365.25
    df2["sin_doy"],  df2["cos_doy"]  = np.sin(theta),     np.cos(theta)       # one wave per year
    df2["sin2_doy"], df2["cos2_doy"] = np.sin(2 * theta), np.cos(2 * theta)   # two waves (K=2 check)

    return df2

In [ ]:
# add sta_mean, sta_std, z (target) and sin_doy/cos_doy (season features)
river_feat = add_features(river_df, cutoff = cutoff_val)


# pairplot 
sample = (river_feat[river_feat["date"] <= pd.Timestamp(cutoff_val)]  # training dates only
    .dropna(subset = ["z"])  
    .sample(5000, random_state = 0))

sample["month"] = sample["date"].dt.month

sns.pairplot(sample, vars = ["z", "sin_doy", "cos_doy", "Latitude", "Longitude"], hue = "month")
plt.show()

In [ ]:
# Baseline model: all features 

num_cols = ["sin_doy", "cos_doy", "Latitude", "Longitude", "sta_mean", "sta_std", "daily_range"]
cat_cols = ["County"]

X_train_all, y_train_all, X_test_all, y_test_all = make_split(river_feat, 
                                                              feature_cols = num_cols + cat_cols, target = "z",
                                                              cutoff = cutoff_val, end = end_val)

val_df_all = river_feat.loc[X_test_all.index]

preprocess_all = ColumnTransformer([("num", StandardScaler(), num_cols),
                                ("cat", OneHotEncoder(drop = "first"), cat_cols)])
lm_all = make_pipeline(preprocess_all, LinearRegression()).fit(X_train_all, y_train_all)

pred_z_all = lm_all.predict(X_test_all)
pred_m_all = pred_z_all * val_df_all["sta_std"] + val_df_all["sta_mean"]

print("RMSE (z):", round(root_mean_squared_error(y_test_all, pred_z_all), 2),
      "\nRMSE (m):", round(root_mean_squared_error(val_df_all["avg"], pred_m_all), 2),
      "\nMAE (m): ", round(mean_absolute_error(val_df_all["avg"], pred_m_all), 2))

In [ ]:
# split into train/test: predict z (standardized level) from season only
X_train_season, y_train_season, X_test_season, y_test_season = make_split(river_feat, 
                                                                          feature_cols = ["sin_doy", "cos_doy"], target = "z",
                                                                          cutoff = cutoff_val, end = end_val) 

val_df_season = river_feat.loc[X_test_season.index]

lm_season_raw = LinearRegression().fit(X_train_season, y_train_season)  # unscaled 
lm_season = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train_season, y_train_season)

w_df_season_raw = get_coefs(lm_season_raw, plot = True, figsize = (6, 4), figtitle = "Baseline: seasonal linear regression")
print("lm_sc_raw:", w_df_season_raw)

pred_z_season = lm_season.predict(X_test_season)
pred_m_season = pred_z_season * val_df_season["sta_std"] + val_df_season["sta_mean"]

print("RMSE (z):", round(root_mean_squared_error(y_test_season, pred_z_season), 2))
print("RMSE (m):", round(root_mean_squared_error(val_df_season["avg"], pred_m_season), 2))
print("MAE (m): ", round(mean_absolute_error(val_df_season["avg"], pred_m_season), 2))

In [ ]:
for i in [0.0001, 0.001, 0.01, 0.1]:
    
    lasso_rg = make_pipeline(preprocess_all, 
                             Lasso(alpha = i)).fit(X_train_all, y_train_all)
    predicted_z  = lasso_rg.predict(X_test_all)
    no_of_zeros  = (lasso_rg[-1].coef_ == 0).sum()
    
    print(i, round(root_mean_squared_error(y_test_all, predicted_z), 3), 
          no_of_zeros, "zeros of", lasso_rg[-1].coef_.size)

In [ ]:
lasso_best = make_pipeline(preprocess_all, Lasso(alpha = 0.01)).fit(X_train_all, y_train_all)

w_las = get_coefs(lasso_best)
print(w_las[w_las["coef"] != 0])

In [ ]:
for i in [1, 100, 10_000, 1_000_000]:
    
    ridge_rg = make_pipeline(preprocess_all, Ridge(alpha = i)).fit(X_train_all, y_train_all)
    predicted_z  = ridge_rg.predict(X_test_all)
    weight   = ridge_rg[-1].coef_
    
    print(i, round(root_mean_squared_error(y_test_all, predicted_z), 3),
          (weight == 0).sum(), "zeros |", round(np.abs(weight).sum(), 3), "Σ|coef|")

In [ ]:
ridge_best = make_pipeline(preprocess_all, Ridge(alpha = 100)).fit(X_train_all, y_train_all)

w_rid = get_coefs(ridge_best)
print(w_rid[w_rid["coef"] != 0].head())

In [ ]:
poly_cols = ["sin_doy", "cos_doy", "daily_range"]

X_train_p, y_train_p, X_test_p, y_test_p = make_split(river_feat2, 
                                                      feature_cols = poly_cols, target = "z", 
                                                      cutoff = cutoff_val, end = end_val)

for m in [1, 2, 3]:
    poly_reg = make_pipeline(PolynomialFeatures(degree = m, include_bias = False),
                             StandardScaler(),
                             LinearRegression()).fit(X_train_p, y_train_p)
    predicted_z = poly_reg.predict(X_test_p)
    print(m, round(root_mean_squared_error(y_test_p, predicted_z), 3), poly_reg[0].n_output_features_, "features")

In [ ]:
poly_best = make_pipeline(PolynomialFeatures(degree = 2, include_bias = False),
                          StandardScaler(), LinearRegression()).fit(X_train_p, y_train_p)

val_df_p  = river_feat.loc[X_test_p.index]

w_p = get_coefs(poly_best)
print(w_p[w_p["coef"] != 0].head())